# Portfolio-IC Upward Weight Sweep

Full PIT run for `portfolio_ic_weight=0.75` and `portfolio_ic_weight=1.0`.

The first executable lines reject T4 runtimes via `nvidia-smi` before any training begins. Use the G4 GPU runtime or another non-T4 higher-tier GPU, then run the notebook.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import pandas as pd

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    drive = None
    userdata = None
    IN_COLAB = False


def run_capture(cmd, check=False):
    return subprocess.run(cmd, text=True, capture_output=True, check=check)


def detect_gpu_name() -> str:
    proc = run_capture(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'])
    if proc.returncode != 0:
        raise RuntimeError('nvidia-smi failed. Attach a GPU runtime before launching training.\n' + proc.stderr)
    name = proc.stdout.strip().splitlines()[0].strip() if proc.stdout.strip() else ''
    if not name:
        raise RuntimeError('No GPU name returned by nvidia-smi.')
    return name


GPU_NAME = detect_gpu_name()
print('GPU:', GPU_NAME)
upper_gpu = GPU_NAME.upper()
if 'T4' in upper_gpu:
    raise RuntimeError(f'Refusing runtime GPU {GPU_NAME}. This sweep requires a non-T4 G4-class runtime.')
allowed_markers = ('G4', 'L4', 'A100', 'H100', 'V100', 'RTX PRO', 'BLACKWELL')
if not any(marker in upper_gpu for marker in allowed_markers):
    raise RuntimeError(
        f'GPU {GPU_NAME} is not in the allowed non-T4 set {allowed_markers}. '
        'Switch to a G4-class or better Colab runtime before continuing.'
    )
print('GPU gate passed.')

REPO_URL = 'https://github.com/magilliam27/MCI-GRU.git'
BRANCH = 'codex/portfolio-ic-hybrid-testing'
REPO_DIR = Path('/content/MCI-GRU') if IN_COLAB else Path.cwd()

if IN_COLAB:
    drive.mount('/content/drive')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

if IN_COLAB and not os.environ.get('FRED_API_KEY'):
    try:
        secret = userdata.get('FRED_API_KEY') if userdata is not None else None
        if secret:
            os.environ['FRED_API_KEY'] = secret
            print('FRED_API_KEY loaded from Colab Secrets.')
    except Exception as exc:
        print('Could not read FRED_API_KEY from Colab Secrets:', exc)
if not os.environ.get('FRED_API_KEY'):
    raise RuntimeError('FRED_API_KEY is required for the current regime-enabled preset.')

drive_data_dir = Path('/content/drive/MyDrive/MCI_GRU_shared/data') if IN_COLAB else REPO_DIR / 'data/raw/market'
drive_market_csv = drive_data_dir / 'sp500_pit_union_lseg_20150101_20260513.csv'
drive_pit_csv = drive_data_dir / 'sp500_pit_joiner_leaver_20160101_20260513_pit_universe.csv'
if not drive_market_csv.exists():
    raise FileNotFoundError(f'Missing market CSV: {drive_market_csv}')
if not drive_pit_csv.exists():
    raise FileNotFoundError(f'Missing PIT universe CSV: {drive_pit_csv}')
repo_market_csv = REPO_DIR / 'data/raw/market/sp500_pit_union_lseg_20150101_20260513.csv'
repo_pit_csv = REPO_DIR / 'data/raw/constituents/sp500_pit_joiner_leaver_20160101_20260513_pit_universe.csv'
repo_market_csv.parent.mkdir(parents=True, exist_ok=True)
repo_pit_csv.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(drive_market_csv, repo_market_csv)
shutil.copy2(drive_pit_csv, repo_pit_csv)

SMOKE_MODE = False
YEARS = [2022, 2023, 2024, 2025]
BASE_SEEDS = [314159, 271828, 161803]
NUM_MODELS = 20
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15
RUN_TAG = datetime.utcnow().strftime('%Y%m%d_%H%M%S_static_regime_full')
RUN_ROOT = (Path('/content/drive/MyDrive/MCI-GRU-Ablations/portfolio_ic_hybrid_upward_sweep') if IN_COLAB else REPO_DIR / 'results' / 'portfolio_ic_hybrid_upward_sweep') / RUN_TAG
TRAINING_OUTPUT_DIR = RUN_ROOT / 'training_runs'
SUMMARY_DIR = RUN_ROOT / 'summaries'
TRAINING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

FROZEN_RECIPE_ID = 'static-threshold-shuffle__pure-ic-returns-5d-val-ic__regime-current-only__ensemble__drop-edge-0p1'
OBJECTIVE_VARIANTS = {
    'portfolio_ic_weight75': {'portfolio_ic_weight': 0.75},
    'portfolio_ic_weight100': {'portfolio_ic_weight': 1.0},
}
EXPECTED_JOB_COUNT = len(YEARS) * len(BASE_SEEDS) * len(OBJECTIVE_VARIANTS)
EXPECTED_TOTAL_MODELS = EXPECTED_JOB_COUNT * NUM_MODELS
assert EXPECTED_JOB_COUNT == 24
assert EXPECTED_TOTAL_MODELS == 480

BASE_OVERRIDES = [
    'data.source=csv',
    'features=with_momentum',
    'features.include_momentum=true',
    'features.include_weekly_momentum=true',
    'features.momentum_encoding=binary',
    'features.momentum_blend_mode=static',
    'features.momentum_blend_fast_weight=0.5',
    'features.include_global_regime=true',
    'features.regime_strict=true',
    'features.regime_enforce_lag_days=0',
    'features.regime_include_subsequent_returns=false',
    'features.regime_change_months=12',
    'features.regime_norm_months=120',
    'features.regime_exclusion_months=1',
    'features.regime_similarity_quantile=0.2',
    'features.regime_min_history_months=24',
    'graph.judge_value=0.8',
    'graph.update_frequency_months=0',
    'graph.corr_lookback_days=252',
    'graph.top_k=0',
    'graph.top_k_metric=corr',
    'graph.use_multi_feature_edges=true',
    'graph.append_snapshot_age_days=false',
    'graph.use_lead_lag_features=false',
    'graph.drop_edge_p=0.1',
    'training.lr_scheduler=cosine',
    'training.learning_rate=5e-5',
    f'training.num_epochs={NUM_EPOCHS}',
    f'training.num_models={NUM_MODELS}',
    f'training.early_stopping_patience={EARLY_STOPPING_PATIENCE}',
    'training.label_type=returns',
    'training.shuffle_train=true',
    'model.label_t=5',
    'model.temporal_encoder=gru_attn',
    'tracking.enabled=false',
    'tracking.log_artifacts=false',
    'tracking.log_checkpoints=false',
    'tracking.log_predictions=false',
    f'data.filename={repo_market_csv.relative_to(REPO_DIR).as_posix()}',
    f'data.pit_universe_csv={repo_pit_csv.relative_to(REPO_DIR).as_posix()}',
    'data.use_pit_universe=true',
    'data.pit_universe_mode=masked_panel',
    'data.pit_min_scoreable_stocks=450',
    'data.pit_breadth_policy=error',
]

jobs = []
for variant_name, variant in OBJECTIVE_VARIANTS.items():
    for year in YEARS:
        for base_seed in BASE_SEEDS:
            name = f'{variant_name}_{year}_seed{base_seed}'
            jobs.append({
                'year': year,
                'base_seed': base_seed,
                'variant': variant_name,
                'portfolio_ic_weight': variant['portfolio_ic_weight'],
                'name': name,
                'overrides': [
                    f'+experiment=pit_temporal_{year}',
                    *BASE_OVERRIDES,
                    'training.loss_type=portfolio_ic',
                    'training.selection_metric=val_loss',
                    'training.portfolio_ic_top_k=10',
                    f"training.portfolio_ic_weight={variant['portfolio_ic_weight']}",
                    'training.portfolio_ic_temperature=0.25',
                    f'seed={base_seed}',
                    f'experiment_name={name}',
                    f'output_dir={TRAINING_OUTPUT_DIR.as_posix()}',
                ],
            })

manifest_path = RUN_ROOT / 'portfolio_ic_upward_sweep_manifest.json'
manifest_path.write_text(json.dumps({
    'recipe_id': FROZEN_RECIPE_ID,
    'branch': BRANCH,
    'run_root': str(RUN_ROOT),
    'gpu_name': GPU_NAME,
    'smoke_mode': SMOKE_MODE,
    'years': YEARS,
    'base_seeds': BASE_SEEDS,
    'num_models': NUM_MODELS,
    'num_epochs': NUM_EPOCHS,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'expected_job_count': EXPECTED_JOB_COUNT,
    'expected_total_models': EXPECTED_TOTAL_MODELS,
    'objective_variants': OBJECTIVE_VARIANTS,
    'jobs': jobs,
}, indent=2), encoding='utf-8')
print('Run root:', RUN_ROOT)
print('Jobs:', len(jobs), 'total models:', EXPECTED_TOTAL_MODELS)
print('Manifest:', manifest_path)


def latest_run_dir(job_name: str) -> Path | None:
    base = TRAINING_OUTPUT_DIR / job_name
    if not base.exists():
        return None
    candidates = sorted(path for path in base.iterdir() if path.is_dir())
    return candidates[-1] if candidates else None


training_results_path = SUMMARY_DIR / 'training_results.csv'
training_results_json = SUMMARY_DIR / 'training_results.json'
training_results = pd.read_csv(training_results_path).to_dict('records') if training_results_path.exists() else []
completed_ok = {row['name'] for row in training_results if row.get('status') == 'OK'}
for job in jobs:
    if job['name'] in completed_ok:
        print('Skipping completed job:', job['name'])
        continue
    print('=' * 100)
    print('Starting:', job['name'], 'weight=', job['portfolio_ic_weight'])
    start = time.time()
    cmd = [sys.executable, '-u', str(REPO_DIR / 'run_experiment.py'), *job['overrides']]
    proc = subprocess.run(cmd, cwd=str(REPO_DIR), text=True)
    elapsed_minutes = round((time.time() - start) / 60.0, 3)
    run_dir = latest_run_dir(job['name'])
    predictions_dir = run_dir / 'averaged_predictions' if run_dir else None
    summary_path = run_dir / 'training_summary.json' if run_dir else None
    eval_path = run_dir / 'evaluation_summary.json' if run_dir else None
    summary = json.loads(summary_path.read_text()) if summary_path and summary_path.exists() else {}
    evaluation = json.loads(eval_path.read_text()) if eval_path and eval_path.exists() else {}
    result = {
        'name': job['name'],
        'variant': job['variant'],
        'portfolio_ic_weight': job['portfolio_ic_weight'],
        'year': job['year'],
        'base_seed': job['base_seed'],
        'loss_type': 'portfolio_ic',
        'selection_metric': 'val_loss',
        'status': 'OK' if proc.returncode == 0 else 'FAILED',
        'returncode': int(proc.returncode),
        'elapsed_minutes': elapsed_minutes,
        'run_dir': str(run_dir) if run_dir else '',
        'predictions_dir': str(predictions_dir) if predictions_dir else '',
        'training_summary.mean_best_val_loss': summary.get('mean_best_val_loss'),
        'training_summary.mean_best_val_ic': summary.get('mean_best_val_ic'),
        'evaluation': evaluation.get('metrics'),
    }
    training_results.append(result)
    pd.DataFrame(training_results).to_csv(training_results_path, index=False)
    training_results_json.write_text(json.dumps(training_results, indent=2), encoding='utf-8')
    print('Return code:', proc.returncode, 'elapsed_minutes:', elapsed_minutes)
    print('Run dir:', result['run_dir'])
    if proc.returncode != 0:
        raise RuntimeError(f"Job failed: {job['name']}")

training_df = pd.DataFrame(training_results)
print('Training results:', training_results_path)
print(training_df[['status', 'variant', 'portfolio_ic_weight', 'year', 'base_seed', 'elapsed_minutes', 'run_dir', 'predictions_dir']])

PIT_WINDOWS = {
    2022: {'test_start': '2022-01-22', 'test_end': '2022-12-31'},
    2023: {'test_start': '2023-01-22', 'test_end': '2023-12-31'},
    2024: {'test_start': '2024-01-22', 'test_end': '2024-12-31'},
    2025: {'test_start': '2025-01-22', 'test_end': '2025-12-31'},
}
BACKTEST_SUFFIX = '_pit_daily_tc_rank_gate'
backtest_results_path = SUMMARY_DIR / 'portfolio_ic_weight75_100_pit_daily_tc_rank_gate_results.csv'
backtest_results_json = SUMMARY_DIR / 'portfolio_ic_weight75_100_pit_daily_tc_rank_gate_results.json'
ok_training = training_df[training_df['status'].eq('OK')].copy()
assert len(ok_training) == EXPECTED_JOB_COUNT, f'Expected {EXPECTED_JOB_COUNT} OK training rows, got {len(ok_training)}'
backtest_rows = pd.read_csv(backtest_results_path).to_dict('records') if backtest_results_path.exists() else []
completed_bt = {row['name'] for row in backtest_rows if row.get('status') == 'OK'}
for row in ok_training.to_dict('records'):
    if row['name'] in completed_bt:
        print('Skipping completed backtest:', row['name'])
        continue
    year = int(row['year'])
    window = PIT_WINDOWS[year]
    predictions_dir = Path(row['predictions_dir'])
    run_dir = Path(row['run_dir'])
    if not predictions_dir.is_dir():
        raise FileNotFoundError(f'Missing predictions directory: {predictions_dir}')
    logs_dir = SUMMARY_DIR / 'logs' / 'backtest' / row['variant'] / f"{year}_seed{int(row['base_seed'])}"
    logs_dir.mkdir(parents=True, exist_ok=True)
    stdout_path = logs_dir / 'stdout.log'
    stderr_path = logs_dir / 'stderr.log'
    cmd = [
        sys.executable, '-X', 'utf8', str(REPO_DIR / 'tests' / 'backtest_sp500_daily.py'),
        '--predictions_dir', str(predictions_dir),
        '--data_file', str(repo_market_csv),
        '--pit_universe_csv', str(repo_pit_csv),
        '--test_start', window['test_start'],
        '--test_end', window['test_end'],
        '--top_k', '10',
        '--label_t', '5',
        '--num_tests', '1',
        '--adjustment_method', 'bhy',
        '--auto_save',
        '--backtest_suffix', BACKTEST_SUFFIX,
        '--transaction_costs',
        '--spread', '10',
        '--slippage', '5',
        '--enable_rank_drop_gate',
        '--min_rank_drop', '30',
    ]
    print('=' * 100)
    print('Backtesting:', row['name'])
    env = os.environ.copy()
    env['MPLBACKEND'] = 'Agg'
    env['PYTHONUTF8'] = '1'
    proc = subprocess.run(cmd, cwd=str(REPO_DIR), text=True, capture_output=True, env=env)
    stdout_path.write_text(proc.stdout, encoding='utf-8', errors='replace')
    stderr_path.write_text(proc.stderr, encoding='utf-8', errors='replace')
    result_csv = run_dir / f'backtest{BACKTEST_SUFFIX}' / 'backtest_results.csv'
    out = {
        'name': row['name'],
        'variant': row['variant'],
        'portfolio_ic_weight': row['portfolio_ic_weight'],
        'year': year,
        'base_seed': int(row['base_seed']),
        'status': 'OK' if proc.returncode == 0 else 'FAILED',
        'returncode': int(proc.returncode),
        'run_dir': str(run_dir),
        'predictions_dir': str(predictions_dir),
        'backtest_dir': str(run_dir / f'backtest{BACKTEST_SUFFIX}'),
        'stdout_log': str(stdout_path),
        'stderr_log': str(stderr_path),
        'scenario.transaction_costs_enabled': True,
        'scenario.spread_bps': 10.0,
        'scenario.slippage_bps': 5.0,
        'scenario.rank_gate_enabled': True,
        'scenario.min_rank_drop': 30,
    }
    if result_csv.exists():
        result_df = pd.read_csv(result_csv)
        if len(result_df):
            out.update({f'backtest.{k}': v for k, v in result_df.iloc[0].to_dict().items()})
    backtest_rows.append(out)
    pd.DataFrame(backtest_rows).to_csv(backtest_results_path, index=False)
    backtest_results_json.write_text(json.dumps(backtest_rows, indent=2), encoding='utf-8')
    print('Return code:', proc.returncode)
    if proc.returncode != 0:
        raise RuntimeError(f"Backtest failed: {row['name']}")

backtest_df = pd.DataFrame(backtest_rows)
ok = backtest_df[backtest_df['status'].eq('OK')].copy()
assert len(ok) == EXPECTED_JOB_COUNT, f'Expected {EXPECTED_JOB_COUNT} OK backtests, got {len(ok)}'
metric_cols = [
    'backtest.total_return',
    'backtest.ARR',
    'backtest.ASR',
    'backtest.MDD',
    'backtest.avg_daily_turnover',
    'backtest.total_transaction_cost',
    'backtest.cost_drag_ARR',
    'backtest.gross_total_return',
    'backtest.net_total_return',
]
available_metrics = [col for col in metric_cols if col in ok.columns]
summary = ok.groupby(['variant', 'portfolio_ic_weight'], dropna=False)[available_metrics].mean(numeric_only=True).reset_index()
summary_path = SUMMARY_DIR / 'portfolio_ic_weight75_100_metric_summary_by_variant.csv'
summary.to_csv(summary_path, index=False)
year_summary = ok.groupby(['variant', 'portfolio_ic_weight', 'year'], dropna=False)[available_metrics].mean(numeric_only=True).reset_index()
year_summary_path = SUMMARY_DIR / 'portfolio_ic_weight75_100_metric_summary_by_year.csv'
year_summary.to_csv(year_summary_path, index=False)
print('Run root:', RUN_ROOT)
print('Backtest results:', backtest_results_path)
print('Summary by variant:', summary_path)
print('Summary by year:', year_summary_path)
print(summary)
print(year_summary)